# Reproduce the results

Rebuilds the tables and figures of the paper from the stored results. No EEG data or GPU needed; takes about two minutes.

In [ ]:
!git clone -q https://github.com/Atytmr07/eeg-fusion-benchmark.git
%cd eeg-fusion-benchmark

## Tables

In [ ]:
# --device cuda reads the stored GPU runs; it does not need a GPU
!python -m src.make_results --device cuda
!python -m src.controls_stats

In [ ]:
import pandas as pd
pd.read_csv('results_v2/results/controls_chbmit.csv').round(3)

## Figures

In [ ]:
!python -m src.make_figures --device cuda
!python -m src.make_supplementary

In [ ]:
from IPython.display import Image, display
for n in ['figure2_performance', 'figure3_equivalence', 'figure4_rank_stability',
          'figure5_variance', 'figure6_artefact_removal', 'figure7_seizure_level',
          'figure8_cross_dataset']:
    display(Image(f'paper/figures_manuscript/{n}.png', width=800))

## Check

Compares the new tables with the committed ones. Differences in the last digits come from floating-point rounding.

In [ ]:
import io, subprocess
import numpy as np, pandas as pd

git = lambda *a: subprocess.run(['git', *a], capture_output=True, text=True).stdout
changed = [f for f in git('diff', '--name-only', 'results_v2/results').split() if f.endswith('.csv')]
worst = 0.0
for f in changed:
    old, new = pd.read_csv(io.StringIO(git('show', f'HEAD:{f}'))), pd.read_csv(f)
    num = old.select_dtypes('number').columns
    assert old.drop(columns=num).equals(new.drop(columns=num)), f
    worst = max(worst, np.nanmax(np.abs(old[num].values - new[num].values), initial=0))
print(f'{len(changed)} tables changed, largest difference {worst:.1e}')